# 03 · Reverse-mode autodiff: `backward()` for any expression

*Build Your Own Autograd: a Tiny Automatic Differentiation Engine*, notebook 3 of 4

In notebook 02 you back-propagated through a neuron by calling seven `_backward` closures in an order you worked out yourself. Here the engine works out the order. A **topological sort** lists every node after the values it was computed from, and `backward()` runs the closures in the reverse of that list. Then you will see why gradients must be accumulated with `+=`, why they must be reset between training steps, how to verify any expression against numerical gradients, and why reverse mode is so cheap.

## What you will build

- `topological_order(root)`: every node of a graph exactly once, children before parents;
- `backward_pass(root)`: the engine behind `Value.backward()`;
- `zero_grad(params)`: reset gradients between training steps;
- `gradient_check(f, xs)`: compare `backward()` with central differences for any expression.

**Learning objectives.** By the end you can:

1. explain why the backward pass needs a topological order, and produce one with a depth-first search;
2. implement `backward()` and use it on arbitrary expressions;
3. demonstrate the bug that overwriting gradients causes, and explain why gradients are zeroed between steps;
4. verify whole expressions numerically, and compare the cost of reverse mode with computing each partial derivative separately.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function yourself and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 0:** the Python standard library only: `math` and `random` (imported in the setup cell), `copy` for one checking helper, `time` for the cost comparison and `statistics` for one closed-form answer in "Put it together". The notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- There are no optional cells, nothing is downloaded, and nothing in this notebook is random.

## Setup

Run this cell first. It creates a seeded random generator, defines `practice`, the helper that checks each function you write against a reference, and imports the three other standard-library modules this notebook uses.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import math
import random

rng = random.Random(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    if isinstance(want, (int, float)) and isinstance(got, (int, float)):
        return math.isclose(got, want, rel_tol=1e-9, abs_tol=atol)
    return got == want


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-9):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import copy
import statistics
import time

## 📦 Recap: what you built earlier

The hidden cell below re-defines, under their usual names, everything this notebook needs from notebooks 01 and 02: `central_difference`; the checking helpers `practice_with` and `on_copies`; the `Value` class with `as_value`; the operations `value_add`, `value_mul`, `value_pow`, `value_exp`, `value_log`, `value_tanh` and `value_relu`; and the printing helper `draw`. `Value.backward()` calls `backward_pass`, which you write in this notebook. Run the cell; open it only if you want to look back at the code.

In [ ]:
# 📦 Recap: what you built in notebooks 01 and 02, under the usual names.

def central_difference(f, x, h=1e-6):
    """Estimate f'(x) from two nearby values of f: (f(x + h) - f(x - h)) / (2h)."""
    return (f(x + h) - f(x - h)) / (2 * h)


def practice_with(probe, name, yours, reference, cases, atol=1e-9):
    """Run practice on probe(yours) and probe(reference); return the function to use from now on.

    A probe wraps a function so that it returns plain numbers, lists or dicts
    that practice can compare. You get back your own function once it passes,
    otherwise the reference, exactly as with practice.
    """
    your_probe, reference_probe = probe(yours), probe(reference)
    chosen = practice(name, your_probe, reference_probe, cases, atol=atol)
    return yours if chosen is your_probe else reference


def on_copies(function):
    """Probe: call function on deep copies of the arguments and report whether it changed them."""
    def run(*args):
        copies = copy.deepcopy(args)
        result = function(*copies)
        return {"arguments unchanged": copies == args, "result": result}
    return run


class Value:
    """A number that remembers how it was computed, so that gradients can flow back through it."""

    def __init__(self, data, children=(), op="", label=""):
        self.data = float(data)              # the number itself
        self.grad = 0.0                      # d(final output) / d(this value), filled in by the backward pass
        self._children = tuple(children)     # the Values this one was computed from
        self._op = op                        # the operation that made it, for printing
        self._backward = _no_gradient_to_pass_on   # replaced by the operation that creates this Value
        self.label = label                   # an optional name, for printing

    def __repr__(self):
        name = f"{self.label} | " if self.label else ""
        return f"Value({name}data={self.data:.6g}, grad={self.grad:.6g})"

    # Each operator hands the work to a module-level function, looked up when it runs.
    def __add__(self, other):        # self + other
        return value_add(self, as_value(other))

    def __radd__(self, other):       # other + self, when other is a plain number
        return value_add(as_value(other), self)

    def __mul__(self, other):        # self * other
        return value_mul(self, as_value(other))

    def __rmul__(self, other):       # other * self
        return value_mul(as_value(other), self)

    def __pow__(self, exponent):     # self ** exponent, for a constant int or float exponent
        if isinstance(exponent, Value):
            raise TypeError("the exponent must be a plain int or float, not a Value")
        return value_pow(self, exponent)

    # These four are built from the three above, so they need no rules of their own.
    def __neg__(self):               # -self
        return self * -1.0

    def __sub__(self, other):        # self - other
        return self + (-as_value(other))

    def __rsub__(self, other):       # other - self
        return as_value(other) + (-self)

    def __truediv__(self, other):    # self / other
        return self * as_value(other) ** -1

    def __rtruediv__(self, other):   # other / self
        return as_value(other) * self ** -1

    def exp(self):
        return value_exp(self)

    def log(self):
        return value_log(self)

    def tanh(self):
        return value_tanh(self)

    def relu(self):
        return value_relu(self)

    def backward(self):
        """Fill in .grad for every Value this one depends on."""
        backward_pass(self)


def _no_gradient_to_pass_on():
    """The _backward of a Value that no operation produced (an input): there is nothing to pass on."""


def as_value(x):
    """Wrap a plain number in a Value; leave a Value unchanged."""
    return x if isinstance(x, Value) else Value(x)


def value_add(a, b):
    out = Value(a.data + b.data, (a, b), "+")

    def _backward():
        a.grad += out.grad
        b.grad += out.grad

    out._backward = _backward
    return out


def value_mul(a, b):
    out = Value(a.data * b.data, (a, b), "*")

    def _backward():
        a.grad += b.data * out.grad
        b.grad += a.data * out.grad

    out._backward = _backward
    return out


def value_pow(a, exponent):
    out = Value(a.data ** exponent, (a,), f"**{exponent}")

    def _backward():
        a.grad += exponent * a.data ** (exponent - 1) * out.grad

    out._backward = _backward
    return out


def value_exp(a):
    out = Value(math.exp(a.data), (a,), "exp")

    def _backward():
        a.grad += out.data * out.grad

    out._backward = _backward
    return out


def value_log(a):
    out = Value(math.log(a.data), (a,), "log")

    def _backward():
        a.grad += out.grad / a.data

    out._backward = _backward
    return out


def value_tanh(a):
    out = Value(math.tanh(a.data), (a,), "tanh")

    def _backward():
        a.grad += (1.0 - out.data ** 2) * out.grad

    out._backward = _backward
    return out


def value_relu(a):
    out = Value(a.data if a.data > 0 else 0.0, (a,), "relu")

    def _backward():
        a.grad += (1.0 if a.data > 0 else 0.0) * out.grad

    out._backward = _backward
    return out


def draw(root):
    """Print the graph below root as a tree: label, operation, data and grad of every node.

    A node that has already been drawn appears again as one line marked "(again)".
    """
    drawn = set()

    def describe(node):
        name = node.label or "·"
        how = node._op or "input"
        return f"{name} [{how}]  data {node.data:+.4f}  grad {node.grad:+.4f}"

    def walk(node, prefix, last, top):
        branch = "" if top else ("└─ " if last else "├─ ")
        again = node in drawn and bool(node._children)
        print(prefix + branch + describe(node) + ("   (again)" if again else ""))
        if again:
            return
        drawn.add(node)
        inner = prefix if top else prefix + ("   " if last else "│  ")
        for index, child in enumerate(node._children):
            walk(child, inner, index == len(node._children) - 1, False)

    walk(root, "", True, True)


def backward_pass(root):
    """Placeholder until section 2 of this notebook, where you write backward_pass."""
    raise NotImplementedError("you write backward_pass in section 2 of this notebook")

### Test graphs, and how the checks work

`build_graph(name)` builds one of five small graphs afresh and returns its output together with a dictionary of its inputs. Your functions in this notebook return lists of `Value`s or change gradients in place, so each check runs them through a **probe** that builds a fresh graph by name and reports plain numbers:

- `order_probe` reports how many nodes your order lists, how many different ones, whether every child comes before its parents and whether the root comes last;
- `backward_probe` reports every input's gradient after your backward pass;
- `zero_grad_probe` reports the gradients and data of a few parameters after your `zero_grad`.

In [ ]:
def build_graph(name):
    """Build one of the small test graphs afresh. Returns (root, inputs), inputs a dict by label."""
    if name == "leaf":                       # a single input and nothing computed from it
        a = Value(3.0, label="a")
        return a, {"a": a}
    if name == "chain":                      # exp(tanh(2a)): a single path
        a = Value(0.5, label="a")
        return (a * 2.0).tanh().exp(), {"a": a}
    if name == "diamond":                    # a·a + exp(a): a is reached along three edges
        a = Value(0.8, label="a")
        return a * a + a.exp(), {"a": a}
    if name == "reuse":                      # u = a·b is used twice, two levels below the root
        a, b = Value(1.5, label="a"), Value(-0.4, label="b")
        u = a * b
        v = u + a
        return (u * v).tanh(), {"a": a, "b": b}
    if name == "neuron":                     # notebook 02's neuron and its loss, helpers included
        x1, x2 = Value(2.0, label="x1"), Value(-1.0, label="x2")
        w1, w2, b = Value(0.4, label="w1"), Value(-0.3, label="w2"), Value(-0.5, label="b")
        loss = ((w1 * x1 + w2 * x2 + b).tanh() - 0.9) ** 2
        return loss, {"x1": x1, "x2": x2, "w1": w1, "w2": w2, "b": b}
    raise ValueError(f"unknown test graph {name!r}")


def order_probe(order_function):
    """Probe for topological_order: order a fresh test graph and report the properties that matter."""
    def run(graph_name):
        root, _ = build_graph(graph_name)
        order = list(order_function(root))
        first_position = {}
        for position, node in enumerate(order):
            first_position.setdefault(node, position)
        children_first = all(
            child in first_position and first_position[child] < first_position[node]
            for node in first_position for child in node._children
        )
        return {"nodes listed": len(order), "different nodes": len(first_position),
                "children before parents": children_first, "root last": bool(order) and order[-1] is root}
    return run


def backward_probe(backward_function):
    """Probe for backward_pass: run it on a fresh test graph and report every input's gradient."""
    def run(graph_name):
        root, inputs = build_graph(graph_name)
        backward_function(root)
        report = {name: node.grad for name, node in inputs.items()}
        report["root"] = root.grad
        return report
    return run


def zero_grad_probe(zero_function):
    """Probe for zero_grad: parameters holding the given gradients in, their grads and data out."""
    def run(*grads):
        params = [Value(10.0 + index) for index in range(len(grads))]
        for param, grad in zip(params, grads):
            param.grad = grad
        zero_function(params)
        return {"grads": [param.grad for param in params], "data": [param.data for param in params]}
    return run

## 1 · Why the order matters

Back-propagation has one rule: a node may call its `_backward` only when its own `grad` is complete, that is, after **every** node that uses it has passed gradient down to it. Equivalently: list the nodes so that each one comes *after* its children (the values it was computed from), which is a **topological order**, and then run the `_backward`s in the *reverse* of that list. In notebook 02 the order of creation happened to work. In general an expression creates helper nodes you never named (inside `a - b`, for example) and reuses values far apart, so the engine has to find the order itself.

**Depth-first search.** To list the graph below `root`, visit a node's children first and then append the node itself. Keep a set of the nodes already visited, so that a node reached along two paths, as in a diamond, is listed only once. The result puts every input before anything computed from it, and `root` last. Below, `a` feeds three edges of the "diamond" graph.

In [ ]:
root, inputs = build_graph("diamond")
draw(root)

### ✍️ Your turn: `topological_order`

Write `topological_order(root)`:

- **input:** the `Value` at the top of a graph;
- **output:** a list of every `Value` reachable from `root` through `_children`, each exactly once, with every node after all of its children: inputs first, `root` last.

Hints: a recursive helper `visit(node)` that returns at once if the node is in a `visited` set, and otherwise adds it, visits each child and then appends the node. (`Value`s can go in a set: Python compares and hashes them by identity.) A loop with an explicit stack works too, and avoids Python's recursion limit on very deep graphs.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def topological_order_reference(root):
    """Every node reachable from root, once each, children before parents (root last)."""
    order, visited = [], set()
    stack = [(root, False)]                 # (node, are its children already listed?)
    while stack:
        node, children_done = stack.pop()
        if children_done:
            order.append(node)
        elif node not in visited:
            visited.add(node)
            stack.append((node, True))      # come back to this node after its children
            for child in node._children:
                stack.append((child, False))
    return order

In [ ]:
def topological_order(root):
    """Every node reachable from root, once each, children before parents (root last)."""
    # ✍️ Your code here (about eight lines)
    raise NotImplementedError

In [ ]:
topological_order = practice_with(order_probe, "topological_order", topological_order, topological_order_reference, cases=[
    ("chain",),
    ("diamond",),
    ("reuse",),
    ("leaf",),                     # edge case: a graph of one node
])
print([node.label or node._op for node in topological_order(build_graph("diamond")[0])])

**Expected behaviour.** ✅ on all four graphs. The diamond's order lists `a` once, before `*` and `exp`, and `+` last, for example `['a', 'exp', '*', '+']` (another valid order, such as `['a', '*', 'exp', '+']`, is just as good; the check accepts any order with the right properties). For the single-node graph the order is just `[a]`.

## 2 · `backward()`

With the order in hand, the backward pass is three lines: set `root.grad = 1.0`, because $\partial\,\text{root} / \partial\,\text{root} = 1$, then call `_backward()` on every node in **reverse** topological order. Each node then sends its complete gradient to its children, which receive everything before their own turn comes.

`backward_pass` does **not** reset any gradient first. That is `zero_grad`'s job (section 4), and keeping the two separate is what lets frameworks accumulate gradients on purpose.

In [ ]:
root, inputs = build_graph("chain")
order = topological_order(root)
names = {node: node.label or node._op or f"{node.data:g}" for node in order}   # a constant shows its value
print("topological order:", [names[node] for node in order])
print("backward order:   ", [names[node] for node in reversed(order)])

### ✍️ Your turn: `backward_pass`

Write `backward_pass(root)`:

- **input:** the `Value` to differentiate, normally a loss;
- **effect:** every node reachable from `root` has $\partial\,\text{root} / \partial\,\text{node}$ added to its `grad`. It returns nothing.

Steps: set `root.grad = 1.0`; then call `node._backward()` for every `node` in `reversed(topological_order(root))`.

Hint: the check compares the inputs' gradients on four graphs, including one where a node is reused two levels down, so a forward loop or a missing seed shows up as a mismatch.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def backward_pass_reference(root):
    root.grad = 1.0
    for node in reversed(topological_order_reference(root)):
        node._backward()

In [ ]:
def backward_pass(root):
    """Seed root.grad = 1.0, then run every _backward in reverse topological order."""
    # ✍️ Your code here (three lines)
    raise NotImplementedError

In [ ]:
backward_pass = practice_with(backward_probe, "backward_pass", backward_pass, backward_pass_reference, cases=[
    ("chain",),
    ("reuse",),
    ("neuron",),
    ("leaf",),                     # edge case: the gradient of a value with respect to itself is 1
])
loss, inputs = build_graph("neuron")
loss.backward()                    # Value.backward() calls backward_pass(loss)
for name, node in inputs.items():
    print(f"dL/d{name:<2} = {node.grad:+.6f}")

**Expected behaviour.** ✅, then the five gradients of notebook 02's neuron: $\partial L / \partial w_1 \approx -1.033070$, $\partial L / \partial w_2 \approx +0.516535$, $\partial L / \partial b \approx -0.516535$, $\partial L / \partial x_1 \approx -0.206614$ and $\partial L / \partial x_2 \approx +0.154960$. This time the graph was built with `- 0.9`, helper nodes and all, and one call to `loss.backward()` found every node and the right order by itself.

## 3 · Why gradients accumulate: the `=` bug

When a value feeds several operations, the loss depends on it along several paths, and the chain rule **adds** their contributions. Each operation adds its own contribution inside its own `_backward`, so every closure must use `+=`. Below is a multiplication written with `=` instead. On `x * x` it reports $x$ instead of $2x$. On an expression where `x` feeds two different multiplications, only the contribution that happens to arrive last survives.

In [ ]:
def value_mul_overwrite(a, b):
    """A deliberately broken multiplication: = instead of +=."""
    out = Value(a.data * b.data, (a, b), "*")

    def _backward():
        a.grad = b.data * out.grad          # BUG: replaces whatever gradient a already has
        b.grad = a.data * out.grad

    out._backward = _backward
    return out


x = Value(3.0, label="x")
value_mul_overwrite(x, x).backward()
print(f"x·x with '=':    dx = {x.grad}   (the true derivative, 2x, is {2 * x.data})")

x = Value(3.0, label="x")
(value_mul_overwrite(x, Value(4.0)) + value_mul_overwrite(x, Value(2.0))).backward()
print(f"4x + 2x with '=': dx = {x.grad}   (the true derivative is 6.0)")

**What to notice.** Both answers are wrong, and neither raises an error: the bug is silent. With `x * x`, the closure sets `a.grad` and then `b.grad`, and since both are the same node the second assignment wipes out the first. With `4x + 2x` you see 4.0 or 2.0, depending on which multiplication's `_backward` runs last. Only `+=` gives the sum, 6.0.

## 4 · Zeroing gradients between steps

The same `+=` has a second consequence: `backward()` adds into whatever `grad` already holds. In training, every step builds a **new** graph from the same parameters, so the parameters' `grad`s still hold the previous step's gradients unless you reset them. That is why every training loop starts by zeroing them (in PyTorch, `optimizer.zero_grad()`). Only the parameters need it: the other nodes are created afresh, with `grad = 0.0`, by every forward pass.

Calling `backward()` twice on the **same** graph is a different mistake: the intermediate nodes still hold their gradients from the first call, so the result is not even a clean multiple. Build a new graph for every backward pass.

In [ ]:
w = Value(0.4, label="w")


def one_step_loss(w):
    """A small loss of w, built as a new graph on every call."""
    return ((w * 2.0 + (-0.5)).tanh() + (-0.9)) ** 2


one_step_loss(w).backward()
print(f"after one step:          dL/dw = {w.grad:+.6f}")
one_step_loss(w).backward()                 # a second step without zeroing
print(f"after a second step:     dL/dw = {w.grad:+.6f}   (twice the true value)")

### ✍️ Your turn: `zero_grad`

Write `zero_grad(params)`:

- **input:** a list of `Value`s, usually a model's parameters;
- **effect:** sets each one's `grad` to `0.0` and leaves its `data` alone. It returns nothing.

Hint: a two-line loop. Assigning a new list to `params` would change nothing in the `Value`s themselves.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def zero_grad_reference(params):
    """Reset the gradient of every Value in params to 0.0."""
    for param in params:
        param.grad = 0.0

In [ ]:
def zero_grad(params):
    """Reset the gradient of every Value in params to 0.0."""
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
zero_grad = practice_with(zero_grad_probe, "zero_grad", zero_grad, zero_grad_reference, cases=[
    (1.5, -2.0, 0.0),
    (7.0,),
    (),                            # edge case: no parameters at all
])
zero_grad([w])
one_step_loss(w).backward()
print(f"zero_grad, then one step: dL/dw = {w.grad:+.6f}")

**Expected behaviour.** ✅, and after `zero_grad` the gradient is back to the single-step value, $\partial L / \partial w \approx -2.228$, half of the doubled value above. In a training loop the order is always: `zero_grad`, forward pass, `backward()`, update.

## 5 · Verify whole expressions: `gradient_check`

Each rule passed its own check in notebook 02, and `backward_pass` passed its check here, but a bug can still hide in how the pieces combine. The final test treats the engine as a black box. Build an expression from several inputs, run `backward()`, and compare every input's gradient with a central difference, which uses only forward passes. Report the largest absolute gap. For well-scaled expressions a correct engine gives gaps around $10^{-9}$ or smaller; a wrong rule gives a gap about as large as the gradient itself.

In [ ]:
def wobble(a, b):
    return (a * b).tanh() + (a ** 2 + 1.0).log() - b / a


a0, b0 = 0.7, -1.3
print("d wobble / da by central difference:", central_difference(lambda t: wobble(Value(t), Value(b0)).data, a0))

### ✍️ Your turn: `gradient_check`

Write `gradient_check(f, xs, h=1e-6)`:

- **inputs:** `f`, a function that takes `len(xs)` `Value`s and returns a `Value`; `xs`, a list of floats; `h`, the step;
- **output:** the largest absolute difference, over all inputs, between the gradient that `backward()` finds and the central-difference estimate.

Steps: (1) make `leaves = [Value(x) for x in xs]`, call `f(*leaves).backward()` and read each `leaf.grad`; (2) for each index `i`, define a function of one float that rebuilds the expression from fresh `Value`s with `xs[i]` replaced, and returns its `.data`; (3) compare with `central_difference(that_function, xs[i], h)`.

Hint: build fresh `Value`s for every forward pass of the numerical part; reusing the leaves would mix their gradients in. Do not change `xs`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def gradient_check_reference(f, xs, h=1e-6):
    """Largest gap between backward() and central differences for f(*Values) at the point xs."""
    leaves = [Value(x) for x in xs]
    f(*leaves).backward()
    worst = 0.0
    for index in range(len(xs)):
        def along(t):
            shifted = list(xs)
            shifted[index] = t
            return f(*[Value(s) for s in shifted]).data
        worst = max(worst, abs(leaves[index].grad - central_difference(along, xs[index], h)))
    return worst

In [ ]:
def gradient_check(f, xs, h=1e-6):
    """Largest gap between backward() and central differences for f(*Values) at the point xs."""
    # ✍️ Your code here (about nine lines)
    raise NotImplementedError

In [ ]:
def broken_square(x):
    return value_mul_overwrite(x, x)      # the '=' bug from section 3: backward() says x, not 2x


gradient_check = practice_with(on_copies, "gradient_check", gradient_check, gradient_check_reference, cases=[
    (wobble, [0.7, -1.3]),
    (broken_square, [1.5]),               # a broken engine must give a large gap, here 1.5
    (lambda a, b, c: a * b + b * c + c * a, [0.2, -0.5, 1.1]),
], atol=1e-6)

expressions = {
    "wobble": (wobble, [0.7, -1.3]),
    "neuron loss": (lambda x1, x2, w1, w2, b: ((w1 * x1 + w2 * x2 + b).tanh() - 0.9) ** 2, [2.0, -1.0, 0.4, -0.3, -0.5]),
    "softplus": (lambda a: (a.exp() + 1.0).log(), [0.3]),
    "ratio of sums": (lambda a, b, c: (a + b + c) / (a * b * c), [1.2, 0.7, -2.0]),
    "relu mix": (lambda a, b: (a - b).relu() * a + (b - a).relu() * b, [0.9, -0.4]),
}
for name, (function, point) in expressions.items():
    print(f"{name:<14} largest gap {gradient_check(function, point):.1e}")
print(f"{'broken square':<14} largest gap {gradient_check(broken_square, [1.5]):.1e}")

**Expected behaviour.** ✅, then gaps of about $10^{-9}$ or less for the five expressions, which mix every operation of the engine, and a gap of $1.5$ for `broken_square`, where `backward()` says $x = 1.5$ but the true derivative is $2x = 3$. A gradient checker earns its keep by failing loudly on broken code; that is why the check above includes a broken case. The ReLU expression is checked away from its kink at 0, where a central difference would straddle two slopes.

## 6 · What reverse mode costs

For a function of $n$ inputs, the central-difference gradient needs $2n$ forward passes. Reverse mode needs one forward pass, which records the graph, and one backward pass, which calls each node's `_backward` once: about the cost of a second forward pass, **whatever $n$ is**. Forward-mode differentiation (Exercise 2) is the mirror image: one pass per *input*. A loss has one output and many parameters, so reverse mode wins by a factor that grows with $n$, which is why deep-learning frameworks use it. The price is memory: the whole graph, with every intermediate value, must be kept until the backward pass has run.

`cost_of_gradient(n)` builds a function of $n$ inputs, then times one reverse-mode gradient and one central-difference gradient. The times depend on your computer; the trend does not.

In [ ]:
def chain_of_pairs(*xs):
    """tanh(x0·x1) + tanh(x1·x2) + ...: one output computed from many inputs."""
    total = Value(0.0)
    for left, right in zip(xs, xs[1:]):
        total = total + (left * right).tanh()
    return total


def cost_of_gradient(n_inputs):
    """Nodes per forward pass, and the seconds taken by reverse mode and by central differences."""
    point = [math.sin(index) for index in range(n_inputs)]
    start = time.perf_counter()
    leaves = [Value(x) for x in point]
    out = chain_of_pairs(*leaves)
    out.backward()
    reverse_seconds = time.perf_counter() - start
    nodes = len(topological_order(out))
    start = time.perf_counter()
    for index in range(n_inputs):
        def along(t):
            shifted = list(point)
            shifted[index] = t
            return chain_of_pairs(*[Value(s) for s in shifted]).data
        central_difference(along, point[index])
    numerical_seconds = time.perf_counter() - start
    return nodes, reverse_seconds, numerical_seconds


print(f"{'inputs':>6} {'nodes per pass':>15} {'passes (numerical)':>19} {'reverse mode':>13} {'numerical':>11} {'ratio':>6}")
for n_inputs in (10, 40, 160):
    nodes, reverse_seconds, numerical_seconds = cost_of_gradient(n_inputs)
    print(f"{n_inputs:>6} {nodes:>15} {2 * n_inputs:>19} {reverse_seconds * 1e3:>10.2f} ms "
          f"{numerical_seconds * 1e3:>8.1f} ms {numerical_seconds / reverse_seconds:>5.0f}×")

**What to notice.** The graph grows linearly with $n$ (about four nodes per input), and so does the reverse-mode time: one forward and one backward pass over it. The numerical gradient needs $2n$ passes over a graph whose size also grows with $n$, so its time grows at least like $n^2$, and the ratio in the last column keeps growing with $n$. For a model with a million parameters, central differences are hopeless, and reverse mode costs about the same as evaluating the loss twice.

## 🧪 Experiments

### 🧪 Try it: `=` versus `+=`

In [ ]:
use_overwrite = True    # set to False, then run this cell again
multiply = value_mul_overwrite if use_overwrite else value_mul
x = Value(3.0, label="x")
f = multiply(x, Value(4.0)) + multiply(x, Value(2.0)) + multiply(x, x)    # 4x + 2x + x²
f.backward()
print(f"df/dx = {x.grad}   (the true value is 4 + 2 + 2x = {6 + 2 * x.data})")

**What to notice.** With `use_overwrite = True` the gradient is one of the contributions, not their sum; with `False` it is 12.0. Change the order of the three terms and the wrong answer changes too, which is what makes this bug so confusing in practice.

### 🧪 Try it: the size of the graph

In [ ]:
n_inputs = 100   # try 10, 200 and 400, then run this cell again (400 takes a few seconds)
nodes, reverse_seconds, numerical_seconds = cost_of_gradient(n_inputs)
print(f"{n_inputs} inputs: {nodes} nodes and {nodes} _backward calls; reverse mode {reverse_seconds * 1e3:.1f} ms, "
      f"central differences {numerical_seconds * 1e3:.0f} ms ({numerical_seconds / reverse_seconds:.0f}× slower)")

**What to notice.** Reverse mode calls `_backward` exactly once per node, so its work is fixed by the size of the graph. Doubling $n$ roughly doubles the reverse-mode time but at least quadruples the numerical one.

## 7 · Put it together: fit a line with your engine

Everything combines into a training loop. Fit $y \approx w x + c$ to six points by gradient descent on the mean squared error. Each step runs `zero_grad`, builds the loss (a new graph), calls `backward()` and moves each parameter a little against its gradient. Least squares also has an exact answer, which `statistics.linear_regression` computes, so we can check where gradient descent ends up. The cell also runs `gradient_check` on the loss before training.

In [ ]:
xs_line = [-2.0, -1.0, 0.0, 1.0, 2.0, 3.0]
ys_line = [-3.1, -0.9, 1.2, 2.8, 5.1, 7.0]
LEARNING_RATE, STEPS = 0.1, 150


def line_loss(w, c):
    """Mean squared error of the line w·x + c on the six points: a new graph on every call."""
    total = Value(0.0)
    for x_i, y_i in zip(xs_line, ys_line):
        error = w * x_i + c - y_i
        total = total + error * error
    return total / len(xs_line)


w, c = Value(0.0, label="w"), Value(0.0, label="c")
start_gap = gradient_check(line_loss, [w.data, c.data])
history = []
for step in range(STEPS):
    zero_grad([w, c])
    loss = line_loss(w, c)
    loss.backward()
    for param in (w, c):
        param.data -= LEARNING_RATE * param.grad
    history.append(loss.data)

slope, intercept = statistics.linear_regression(xs_line, ys_line)
print(f"gradient check before training: largest gap {start_gap:.1e}")
print(f"loss {history[0]:.4f} at the start, {history[-1]:.6f} after {STEPS} steps")
print(f"gradient descent: w = {w.data:.5f}, c = {c.data:.5f}")
print(f"least squares:    w = {slope:.5f}, c = {intercept:.5f}")

assert start_gap < 1e-6, "backward() disagrees with central differences on the loss"
assert all(later <= earlier + 1e-12 for earlier, later in zip(history, history[1:])), "the loss should never rise"
assert history[-1] < 0.01 * history[0], "the loss should fall by more than 99%"
assert abs(w.data - slope) < 1e-3 and abs(c.data - intercept) < 1e-3, "gradient descent should reach the least-squares line"

**What to notice.** The loss falls at every step, from about $15.8$ to the least-squares minimum, and the parameters match the exact answer to five decimals. Your engine computed every gradient; nothing was derived by hand. This is the whole training recipe that notebook 04 applies to a neural network.

## Exercises

### Exercise 1 · Kahn's algorithm: the backward order directly

A topological order can also be found without recursion and without a set of visited nodes. Count, for every node, how many times other nodes use it. Start from the root. Repeatedly take a node whose users have all been processed, append it, and lower the count of each of its children (once per use); a child whose count reaches zero is ready. This produces the **backward** order directly: root first, and every node before its children. Write `backward_order_kahn(root)`; you may use `topological_order` to find the nodes and count their uses.

In [ ]:
def backward_order_kahn_reference(root):
    uses = {}                                       # node -> number of times other nodes use it
    for node in topological_order_reference(root):
        for child in node._children:
            uses[child] = uses.get(child, 0) + 1
    ready, order = [root], []
    while ready:
        node = ready.pop()
        order.append(node)
        for child in node._children:
            uses[child] -= 1
            if uses[child] == 0:
                ready.append(child)
    return order


def reverse_order_probe(order_function):
    """Probe for a backward order: parents before children and the root first."""
    def run(graph_name):
        root, _ = build_graph(graph_name)
        order = list(order_function(root))
        first_position = {}
        for position, node in enumerate(order):
            first_position.setdefault(node, position)
        parents_first = all(
            child in first_position and first_position[node] < first_position[child]
            for node in first_position for child in node._children
        )
        return {"nodes listed": len(order), "different nodes": len(first_position),
                "parents before children": parents_first, "root first": bool(order) and order[0] is root}
    return run

In [ ]:
def backward_order_kahn(root):
    """Every node once, root first and each node before its children, by counting uses."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
backward_order_kahn = practice_with(reverse_order_probe, "backward_order_kahn", backward_order_kahn,
                                    backward_order_kahn_reference, cases=[("diamond",), ("reuse",), ("neuron",)])

### Exercise 2 · Forward mode with dual numbers (stretch)

Forward-mode autodiff carries, alongside every value, its derivative with respect to **one chosen input**. Represent such a "dual number" as a tuple `(value, tangent)`. The input you differentiate with respect to starts with tangent 1, every other input with tangent 0, and each operation propagates the tangent by the ordinary rules: $(a, \dot a) \cdot (b, \dot b) = (ab,\ \dot a b + a \dot b)$ and $\tanh(a, \dot a) = (\tanh a,\ (1 - \tanh^2 a)\,\dot a)$. Write `dual_mul(x, y)` and `dual_tanh(x)`. The cell after them computes one derivative of a neuron in forward mode and compares it with `backward()`.

In [ ]:
def dual_mul_reference(x, y):
    (a, a_dot), (b, b_dot) = x, y
    return (a * b, a_dot * b + a * b_dot)


def dual_tanh_reference(x):
    a, a_dot = x
    t = math.tanh(a)
    return (t, (1.0 - t * t) * a_dot)

In [ ]:
def dual_mul(x, y):
    """The product of two dual numbers (value, tangent)."""
    # ✍️ Your code here
    raise NotImplementedError


def dual_tanh(x):
    """tanh of a dual number (value, tangent)."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
dual_mul = practice("dual_mul", dual_mul, dual_mul_reference, cases=[((2.0, 1.0), (-3.0, 0.0)), ((0.5, 2.0), (4.0, -1.0))])
dual_tanh = practice("dual_tanh", dual_tanh, dual_tanh_reference, cases=[((0.5, 1.0),), ((-2.0, 3.0),)])


def dual_add(x, y):
    return (x[0] + y[0], x[1] + y[1])


# d/dw1 of tanh(w1·x1 + w2·x2): one forward pass with tangent 1 on w1 and 0 elsewhere
forward_mode = dual_tanh(dual_add(dual_mul((0.4, 1.0), (2.0, 0.0)), dual_mul((-0.3, 0.0), (-1.0, 0.0))))[1]
vw1, vw2, vx1, vx2 = Value(0.4), Value(-0.3), Value(2.0), Value(-1.0)
(vw1 * vx1 + vw2 * vx2).tanh().backward()
print(f"d/dw1 in forward mode: {forward_mode:+.10f}   in reverse mode: {vw1.grad:+.10f}")

### Exercise 3 · Open questions

1. Forward mode gave one partial derivative per pass. How many forward-mode passes give all four partial derivatives of the neuron above, and how many would a model with a million parameters need? When would forward mode be the better choice?
2. Our engine keeps every intermediate `Value` alive until `backward()` has run. Roughly how many `Value`s does one step of the line fit create? How would that number grow for a network with a million parameters and a batch of 64 examples?
3. Your `topological_order` may be recursive. What happens on a chain of 5,000 operations, and what are two ways to fix it?

## Common errors and debugging

| Symptom | Likely cause | Fix |
|---|---|---|
| every gradient is 0 after `backward()` | `root.grad` was never seeded | set `root.grad = 1.0` first |
| gradients near the root are right, the rest wrong | the `_backward`s ran in topological order instead of reversed | iterate over `reversed(topological_order(root))` |
| some gradients are exactly doubled | a node was listed twice (no visited set), so its `_backward` ran twice | keep a `visited` set |
| a value used twice gets only one contribution | a rule uses `=` instead of `+=` | always accumulate |
| gradients grow from step to step | `zero_grad` is missing, or it runs after `backward()` | zero first, then forward, then `backward()` |
| strange values after calling `backward()` twice | the intermediate nodes still hold the first call's gradients | build a new graph (a new forward pass) for every backward pass |
| `RecursionError: maximum recursion depth exceeded` | a recursive search on a very deep graph (thousands of chained operations) | use an explicit stack, as the reference does |
| `gradient_check` reports a gap about as large as a gradient | a backward rule is wrong; the gap's input tells you where to look | check the rules of the operations that input passes through |

## Summary

- The backward pass must visit each node after every node that uses it: the **reverse** of a topological order.
- A depth-first search with a set of visited nodes lists each node once, children first.
- `backward()` seeds the root with 1 and runs every `_backward` in reverse order; one call fills in every gradient.
- Gradients accumulate. `+=` is what makes reused values correct, and `zero_grad` is what keeps training steps independent.
- `gradient_check` verifies any expression against central differences, and should fail loudly on a broken rule.
- Reverse mode gets all $n$ partial derivatives for about the cost of two forward passes; central differences need $2n$ passes.

You built `topological_order`, `backward_pass`, `zero_grad` and `gradient_check`.

## Next

In **04 · A differentiable neural network** (`04_differentiable_neural_network.ipynb`) you build neurons, layers and a multilayer perceptron from `Value`s, train it on a two-moons dataset with the engine you have written, and map each piece onto PyTorch.